In [ ]:
import os
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')

train_path = 'C:/Users/zhang/mine/finTech/data/A/new/?????'
test_path = 'C:/Users/zhang/mine/finTech/data/A/new/?????'

train_tag_DF = pd.read_csv(train_path + '/?????_tag.csv')
train_trd_DF = pd.read_csv(train_path + '/?????_trd.csv')
train_beh_DF = pd.read_csv(train_path + '/?????_beh.csv')

train_trd_DF.drop_duplicates(inplace = True)
train_beh_DF.drop_duplicates(inplace = True)
# 0: ??
# 1: ??

In [ ]:
train_tag_DF.drop(['edu_deg_cd', 'deg_cd', 'atdd_type'], axis = 1, inplace = True)

# ?? \N ~??
train_tag_DF.loc[train_tag_DF['job_year'] == '\\N', 'job_year'] = '-1'

strColName = ['gdr_cd', 'mrg_situ_cd', 'acdm_deg_cd']
colname = train_tag_DF.columns.values.tolist()
for col in colname[1:]:
    if col in strColName:
        train_tag_DF[col] = train_tag_DF[col].fillna('Unknown')
        train_tag_DF.loc[train_tag_DF[col] == '\\N', col] = 'Unknown'
        train_tag_DF.loc[train_tag_DF[col] == '~', col] = 'Unknown'
    if (not np.issubdtype(train_tag_DF[col], np.int64)):
        if ('0' in train_tag_DF[col].value_counts().index.tolist()) and ('-1' in train_tag_DF[col].value_counts().index.tolist()):
            train_tag_DF.loc[train_tag_DF[col] == '\\N', col] = '-2'
        elif ('0' in train_tag_DF[col].value_counts().index.tolist()):
            train_tag_DF.loc[train_tag_DF[col] == '\\N', col] = '-1'
        else:
            train_tag_DF.loc[train_tag_DF[col] == '\\N', col] = '0'

for col in colname[1:]:
    if (not np.issubdtype(train_tag_DF[col], np.int64)) and (col not in strColName):
        train_tag_DF[col] = train_tag_DF[col].astype(np.int64)

In [ ]:
# ??????ind?????
for col in colname[1:]:
    if col in strColName:
        temp = pd.crosstab(train_tag_DF['id'], train_tag_DF[col])
        temp = temp.reset_index()
        temp_columns = temp.columns.tolist()
        for col_2 in temp_columns[1:]:
            temp.rename(columns = {col_2: '{}_{}'.format(col, col_2)}, inplace = True)
        train_tag_DF = pd.merge(train_tag_DF, temp, on = 'id', how = 'left')
        temp_columns = temp.columns.tolist()
        for col_2 in temp_columns[1:]:
            train_tag_DF[col_2] = train_tag_DF[col_2].fillna(0)
    
    if col.split('_')[-1] == 'ind':
        temp = pd.crosstab(train_tag_DF['id'], train_tag_DF[col])
        temp = temp.reset_index()
        temp_columns = temp.columns.tolist()
        for col_2 in temp_columns[1:]:
            temp.rename(columns = {col_2: '{}_{}'.format(col, col_2)}, inplace = True)
        train_tag_DF = pd.merge(train_tag_DF, temp, on = 'id', how = 'left')
        temp_columns = temp.columns.tolist()
        for col_2 in temp_columns[1:]:
            train_tag_DF[col_2] = train_tag_DF[col_2].fillna(0)

In [ ]:
train_tag_DF['I12_mon_whl_tms'] = train_tag_DF[['l12mon_buy_fin_mng_whl_tms',
                                                'l12_mon_fnd_buy_whl_tms',
                                                'l12_mon_insu_buy_whl_tms',
                                                'l12_mon_gld_buy_whl_tms']].apply(lambda x: x.fillna(0).astype(np.int).sum(), axis = 1).astype(np.int64)
# ???????????????????

train_tag_DF['dnl_ind'] = train_tag_DF[['dnl_mbl_bnk_ind', 'dnl_bind_cmb_lif_ind']].apply(lambda x: x.sum(), axis = 1)
# ??????'dnl_mbl_bnk_ind', 'dnl_bind_cmb_lif_ind',

train_tag_DF['cur_debit_card'] = train_tag_DF[['cur_debit_cnt', 
                                               'cur_debit_min_opn_dt_cnt', 
                                               'cur_debit_crd_lvl']].apply(lambda x: x[0] * x[1] * x[2] / 100, axis = 1)

train_tag_DF['cur_credit_card'] = train_tag_DF[[ 'cur_credit_cnt', 
                                                'cur_credit_min_opn_dt_cnt', 
                                                'hld_crd_card_grd_cd',
                                                'crd_card_act_ind']].apply(lambda x: x[0] * x[1] * (x[2] + 1e-5) * (x[3] + 1e-5), axis = 1)

train_tag_DF['ic_fr_sh_ind'] = train_tag_DF[['ic_ind', 'fr_or_sh_ind']].apply(lambda x: 4 if ((x[0] == 1) & (x[1] == 1)) else (3 if (x[1] == 1) else (2 if (x[0] == 1) else (1 if ((x[0] == 0) & (x[1] == 0)) else 0))), axis = 1)
# 0: no, 1: ic, 2: fr or sh, 3: both

train_tag_DF['hav_car_hou_ind'] = train_tag_DF[['hav_car_grp_ind',
                                                'hav_hou_grp_ind']].apply(lambda x: 4 if ((x[0] == 1) & (x[1] == 1)) else (3 if (x[1] == 1) else (2 if (x[0] == 1) else (1 if ((x[0] == 0) & (x[1] == 0)) else 0))), axis = 1)
# 0: no, 1: Car, 2: House, 3: both

train_tag_DF['ovd_cnt'] = train_tag_DF[['ovd_30d_loan_tot_cnt',
                                        'his_lng_ovd_day']].apply(lambda x: x[0] * x[1], axis = 1)

train_tag_DF['rsk_combine'] = train_tag_DF[['vld_rsk_ases_ind',
                                         'fin_rsk_ases_grd_cd',
                                         'confirm_rsk_ases_lvl_typ_cd',
                                         'cust_inv_rsk_endu_lvl_cd',
                                        ]].apply(lambda x: -abs(x[0]*x[1]*x[2]*x[3]) if any(x<0) else x[0]*x[1]*x[2]*x[3], axis = 1)

train_tag_DF['ast_combine'] = train_tag_DF[['tot_ast_lvl_cd',
                                         'pot_ast_lvl_cd',
                                        ]].apply(lambda x: -abs(x[0]*x[1]) if any(x<0) else x[0]*x[1], axis = 1)

train_tag_DF['rsk_mean'] = train_tag_DF[['vld_rsk_ases_ind',
                                         'fin_rsk_ases_grd_cd',
                                         'confirm_rsk_ases_lvl_typ_cd',
                                         'cust_inv_rsk_endu_lvl_cd',
                                        ]].apply(lambda x: x.mean(), axis = 1)

train_tag_DF['ast_mean'] = train_tag_DF[['tot_ast_lvl_cd',
                                         'pot_ast_lvl_cd',
                                        ]].apply(lambda x: x.mean(), axis = 1)


In [ ]:
tag_id_list = train_tag_DF.id.unique().tolist()
trd_id_list = train_trd_DF.id.unique().tolist()
beh_id_list = train_beh_DF.id.unique().tolist()

print('tag id??: ', len(tag_id_list))
print('trd id??: ', len(trd_id_list))
print('beh id??: ', len(beh_id_list))

print('trd shape: {}, beh shape: {}'.format(train_trd_DF.shape[0], train_beh_DF.shape[0]))
print('trd??id???tag????: ', np.sum(train_trd_DF.id.isin(train_tag_DF.id)))
print('beh??id???tag????', np.sum(train_beh_DF.id.isin(train_tag_DF.id)))

# ?trx???

In [ ]:
trx_cod1_cd = train_trd_DF.groupby(['id', 'Trx_Cod1_Cd'])['trx_tm'].count()
trx_cod1_cd = trx_cod1_cd.reset_index()
trx_cod1_cd.rename(columns = {'trx_tm': 'trx_cod1_tms'}, inplace = True)
trx_cod1_cd.sort_values(['id', 'trx_cod1_tms'], inplace = True)

trx_cod1_cd.drop_duplicates('id', keep = 'last', inplace = True)
trx_cod1_cd.drop(['trx_cod1_tms'], axis = 1, inplace = True)
trx_cod1_cd.rename(columns = {'Trx_Cod1_Cd': 'most_Trx_Cod1_Cd'}, inplace = True)


trx_cod2_cd = train_trd_DF.groupby(['id', 'Trx_Cod2_Cd'])['trx_tm'].count()
trx_cod2_cd = trx_cod2_cd.reset_index()
trx_cod2_cd.rename(columns = {'trx_tm': 'trx_cod2_tms'}, inplace = True)
trx_cod2_cd.sort_values(['id', 'trx_cod2_tms'], inplace = True)

trx_cod2_cd.drop_duplicates('id', keep = 'last', inplace = True)
trx_cod2_cd.drop(['trx_cod2_tms'], axis = 1, inplace = True)
trx_cod2_cd.rename(columns = {'Trx_Cod2_Cd': 'most_Trx_Cod2_Cd'}, inplace = True)

train_DF = pd.merge(train_tag_DF, trx_cod1_cd, on = 'id', how = 'left')
train_DF = pd.merge(train_DF, trx_cod2_cd, on = 'id', how = 'left')

train_DF['most_Trx_Cod1_Cd'] = train_DF['most_Trx_Cod1_Cd'].fillna(0)
train_DF['most_Trx_Cod2_Cd'] = train_DF['most_Trx_Cod2_Cd'].fillna(0)

# ?????
# B: ??, C: ??
id_pay = train_trd_DF.groupby(['id', 'Dat_Flg1_Cd'])['cny_trx_amt'].sum().reset_index()

id_income = id_pay[id_pay['Dat_Flg1_Cd'] == 'C']
id_pay = id_pay[id_pay['Dat_Flg1_Cd'] == 'B']

id_income.reset_index(drop = True, inplace = True)
id_pay.reset_index(drop = True, inplace = True)

id_income.drop(['Dat_Flg1_Cd'], axis = 1, inplace = True)
id_pay.drop(['Dat_Flg1_Cd'], axis = 1, inplace = True)

id_income.rename(columns = {'cny_trx_amt': 'cny_trx_income'}, inplace = True)
id_pay.rename(columns = {'cny_trx_amt': 'cny_trx_pay'}, inplace = True)

id_income['cny_trx_income'] = id_income['cny_trx_income'].abs()
id_pay['cny_trx_pay'] = id_pay['cny_trx_pay'].abs()

####
# ?tag??
train_DF = pd.merge(train_DF, id_income, on = 'id', how = 'left')
train_DF = pd.merge(train_DF, id_pay, on = 'id', how = 'left')

train_DF['cny_trx_income'] = train_DF['cny_trx_income'].fillna(0)
train_DF['cny_trx_pay'] = train_DF['cny_trx_pay'].fillna(0)
# ???\??
train_DF['actual_income'] = train_DF[['cny_trx_income', 'cny_trx_pay']].apply(lambda x: x[0] - x[1], axis = 1)
#train_DF.head()

###
# ????
trx_way = train_trd_DF.groupby(['id', 'Dat_Flg3_Cd'])['trx_tm'].count()
trx_way = trx_way.reset_index()
trx_way.rename(columns = {'trx_tm': 'trx_way_tms'}, inplace = True)
trx_way.sort_values(['id', 'trx_way_tms'], inplace = True)

trx_way.drop_duplicates('id', keep = 'last', inplace = True)
trx_way.drop(['trx_way_tms'], axis = 1, inplace = True)
trx_way.rename(columns = {'Dat_Flg3_Cd': 'most_trx_way'}, inplace = True)
train_DF = pd.merge(train_DF, trx_way, on = 'id', how = 'left')
train_DF['most_trx_way'] = train_DF['most_trx_way'].fillna('Undo')

# ????
trx_tms = train_trd_DF.groupby(['id'])['trx_tm'].count()
trx_tms = trx_tms.reset_index()
trx_tms.rename(columns = {'trx_tm': 'trx_tms'}, inplace = True)
train_DF = pd.merge(train_DF, trx_tms, on = 'id', how = 'left')
train_DF['trx_tms'] = train_DF['trx_tms'].fillna(0)

trx_Dat_Flg1_Cd_cnt = pd.crosstab(train_trd_DF['id'],train_trd_DF['Dat_Flg1_Cd'])
trx_Dat_Flg1_Cd_cnt = trx_Dat_Flg1_Cd_cnt.reset_index()
temp_columns = trx_Dat_Flg1_Cd_cnt.columns.tolist()
for col in temp_columns[1:]:
    trx_Dat_Flg1_Cd_cnt.rename(columns = {col: 'Dat_Flg1_Cd_' + col}, inplace = True)
train_DF = pd.merge(train_DF, trx_Dat_Flg1_Cd_cnt, on = 'id', how = 'left')

temp_columns = trx_Dat_Flg1_Cd_cnt.columns.tolist()
for col in temp_columns[1:]:
    train_DF[col] = train_DF[col].fillna(0)

trx_Dat_Flg3_Cd_cnt = pd.crosstab(train_trd_DF['id'],train_trd_DF['Dat_Flg3_Cd'])
trx_Dat_Flg3_Cd_cnt = trx_Dat_Flg3_Cd_cnt.reset_index()
temp_columns = trx_Dat_Flg3_Cd_cnt.columns.tolist()
for col in temp_columns[1:]:
    trx_Dat_Flg3_Cd_cnt.rename(columns = {col: 'Dat_Flg3_Cd_' + col}, inplace = True)
train_DF = pd.merge(train_DF, trx_Dat_Flg3_Cd_cnt, on = 'id', how = 'left')

temp_columns = trx_Dat_Flg3_Cd_cnt.columns.tolist()
for col in temp_columns[1:]:
    train_DF[col] = train_DF[col].fillna(0)

# ??app???beh

In [ ]:
# ????
beh_page_no_cnt = pd.crosstab(train_trd_DF['id'],train_beh_DF['page_no'])
beh_page_no_cnt = beh_page_no_cnt.reset_index()
page_no_columns = beh_page_no_cnt.columns.tolist()
for col in page_no_columns[1:]:
    beh_page_no_cnt.rename(columns = {col: 'beh_page_no_' + col}, inplace = True)
train_DF = pd.merge(train_DF, beh_page_no_cnt, on = 'id', how = 'left')
page_no_columns = beh_page_no_cnt.columns.tolist()
for col in page_no_columns[1:]:
    train_DF[col] = train_DF[col].fillna(0)
    
## ????????????
id_beh = train_beh_DF.groupby(['id', 'page_no'])['page_tm'].count()
id_beh = id_beh.reset_index()
id_beh.rename(columns = {'page_tm': 'page_tms'}, inplace = True)
id_beh.sort_values(['id', 'page_tms'], inplace = True)

# ???
id_beh_class_count = id_beh.groupby(['id']).count()
id_beh_class_count = id_beh_class_count.reset_index()
id_beh_class_count.drop(['page_tms'], axis = 1, inplace = True)
id_beh_class_count.rename(columns = {'page_no': 'page_type_count'}, inplace = True)

id_beh.drop_duplicates('id', keep = 'last', inplace = True)
id_beh.drop(['page_tms'], axis = 1, inplace = True)
id_beh.rename(columns = {'page_no': 'most_tms_page_no'}, inplace = True)

# ?????????????
beh_count = train_beh_DF.groupby(['id'])['page_tm'].count()
beh_count = beh_count.reset_index()
beh_count.rename(columns = {'page_tm': 'total_page_no_tms'}, inplace = True)

train_DF = pd.merge(train_DF, id_beh_class_count, on = 'id', how = 'left')
train_DF = pd.merge(train_DF, id_beh, on = 'id', how = 'left')
train_DF = pd.merge(train_DF, beh_count, on = 'id', how = 'left')

#train_DF['total_page_no_tms'] = train_DF['total_page_no_tms'].fillna(0)
#train_DF['total_page_no_tms'] = train_DF['total_page_no_tms'].astype(np.int64)
train_DF[['total_page_no_tms', 'page_type_count']] = train_DF[['total_page_no_tms', 'page_type_count']].fillna(0)
train_DF[['total_page_no_tms', 'page_type_count']] = train_DF[['total_page_no_tms', 'page_type_count']].astype(np.int64)

train_DF['most_tms_page_no'] = train_DF['most_tms_page_no'].fillna('Undo')
#train_DF.head()

# ????

In [ ]:
[#'l1y_crd_card_csm_amt_dlm_cd',
 'dnl_ind',
 'gdr_cd_M',
 'crd_card_act_ind_0',
 'cny_trx_income',
 'most_Trx_Cod2_Cd',
 'ovd_cnt',
 'most_trx_way',
 'Dat_Flg1_Cd_C', #### 0.754746591585147
 'acdm_deg_cd_G',
 'most_Trx_Cod1_Cd',
 'fr_or_sh_ind_0',
 'acdm_deg_cd',
 'acdm_deg_cd_F',
 'cny_trx_pay',
 'hav_car_grp_ind_0',
 'dnl_bind_cmb_lif_ind_0',
 'Dat_Flg3_Cd_B',
 'l6mon_daim_aum_cd',
 'cur_credit_min_opn_dt_cnt',
 'page_type_count',
 'hld_crd_card_grd_cd',
 'Dat_Flg3_Cd_A',
 'his_lng_ovd_day',
 'pot_ast_lvl_cd',
 'perm_crd_lmt_cd',
 'rsk_combine',  #### 0.7561045164362351 ? DatFlg1_Cd_C 
 'cur_credit_card',
 'cur_debit_min_opn_dt_cnt',
 'beh_page_no_CQC',# ??
 'beh_page_no_CQA',
 'actual_income',
 'Dat_Flg1_Cd_B',
 'trx_tms',
 'pl_crd_lmt_cd',
 'loan_act_ind',
 'job_year', # 'job_year' 0.7547546933775396,
 'cur_credit_cnt',
 'beh_page_no_MSG',## 0.7546915362232058
 'cust_inv_rsk_endu_lvl_cd',
 'dnl_mbl_bnk_ind', ## ??
 'beh_page_no_AAO',
 'beh_page_no_GBA',
 'beh_page_no_TRN',
 'Dat_Flg3_Cd_C',
 'beh_page_no_MTA', #### 0.7550831381998223 ? DatFlg1_Cd_C, ??7531
 'rsk_mean',
 'age',
 'fin_rsk_ases_grd_cd',
 'hav_car_hou_ind',
 'cur_debit_card',
 'beh_page_no_CQD', #### 0.7551685752832359 ? DatFlg1_Cd_C
 'ast_mean',
 'ast_combine',
 'mrg_situ_cd', #### 0.7558625214214152 ?DatFlg1_Cd_C
 'most_tms_page_no',
 'cur_debit_cnt',
 'beh_page_no_EGB', ## ?????
 'frs_agn_dt_cnt', ### DatFlg1_Cd_C 0.7553716264550773, rsk_mean: 0.7561045164362351, ??725264
 'acdm_deg_cd_C', ## ?????
 'l12_mon_fnd_buy_whl_tms',
 'I12_mon_whl_tms',
 'beh_page_no_SZA', ## ?????
 'l12mon_buy_fin_mng_whl_tms',
 'beh_page_no_CQE', ### DatFlg1_Cd_C 0.7547485249674225, rsk_mean: 0.7561045164362351, ??75395
 'beh_page_no_CQB', #### beh_page_no_CQE??
 'tot_ast_lvl_cd',
 'ovd_30d_loan_tot_cnt', ## ?????
 'gdr_cd', ## DatFlg1_Cd_C, rsk_mean 0.7561285916488566

In [ ]:
#train_DF_weneed = train_DF.drop(['id', 'cny_trx_pay', 'cny_trx_income'], axis = 1)
train_DF_weneed = train_DF.drop(['id',
                                 'Dat_Flg1_Cd_C', #### 0.754746591585147
                                 'rsk_mean', #### 0.7561045164362351 ? DatFlg1_Cd_C
                                 
                                 'vld_rsk_ases_ind_1',
                                 'vld_rsk_ases_ind_0',
                                 'vld_rsk_ases_ind_-1',
                                 'vld_rsk_ases_ind',
                                 'total_page_no_tms',
                                 'mrg_situ_cd_Z',
                                 'mrg_situ_cd_Unknown',
####                                 # 'mrg_situ_cd_O', # 0.7539338805357567
                                 'mrg_situ_cd_B', ### 0.7539528921281328
                                 'mrg_situ_cd_A',
                                 'loan_act_ind_1',
                                 'loan_act_ind_0',
                                 'loan_act_ind_-1',
                                 'l6mon_agn_ind_1',
                                 'l6mon_agn_ind_0',
                                 'l6mon_agn_ind_-1',
                                 'l6mon_agn_ind',
                                 'l12_mon_insu_buy_whl_tms',
                                 'l12_mon_gld_buy_whl_tms',
                                 'ic_ind_1',
                                 'ic_ind_0',
                                 'ic_ind_-1',
                                 'ic_ind',
                                 'ic_fr_sh_ind',
                                 'hav_hou_grp_ind_1',
                                 'hav_hou_grp_ind_0',
                                 'hav_hou_grp_ind_-1',
                                 'hav_hou_grp_ind',
                                 'hav_car_grp_ind_1',
                                 'hav_car_grp_ind_-1',
                                 'hav_car_grp_ind',
                                 'gdr_cd_Unknown',
                                 'gdr_cd_F',
                                 'fr_or_sh_ind_1',
                                 'fr_or_sh_ind_-1',
                                 'fr_or_sh_ind',
                                 'dnl_mbl_bnk_ind_1',
###                                 # 'dnl_mbl_bnk_ind_0', # 0.7538799299636874
                                 'dnl_mbl_bnk_ind_-1',
                                 'dnl_bind_cmb_lif_ind_1',
                                 'dnl_bind_cmb_lif_ind_-1',
                                 'dnl_bind_cmb_lif_ind',
                                 'cur_debit_crd_lvl',
                                 'crd_card_act_ind_1',
                                 'crd_card_act_ind_-1',
                                 'crd_card_act_ind',
                                 'confirm_rsk_ases_lvl_typ_cd',
                                 'bk1_cur_year_mon_avg_agn_amt_cd',
                                 'beh_page_no_ZY1',
###                                 # 'beh_page_no_XAI', #  0.7522612747028713
###                                 # 'beh_page_no_XAG', # 0.753824276173899
                                 'beh_page_no_SZD',
                                 'beh_page_no_SYK',
                                 'beh_page_no_LCT',
                                 'beh_page_no_LC0',
###                                 # 'beh_page_no_JJK', # 0.7537358009183381
                                 'beh_page_no_JJD',
###                                 # 'beh_page_no_JF2', # 0.7530571837396289
###                                 # 'beh_page_no_FTR', # 0.7533822681593851
                                 'beh_page_no_FLS',
                                 'beh_page_no_FDA',
                                 'beh_page_no_EGA',
                                 'beh_page_no_CTR',
                                 'beh_page_no_BWE',
###                                 # 'beh_page_no_BWA', # 0.7539452506648534
                                 'acdm_deg_cd_Z',
                                 'acdm_deg_cd_Unknown',
                                 'acdm_deg_cd_D',
###                                 # 'acdm_deg_cd_31', # 0.7536938649361238
                                 'acdm_deg_cd_30',
                                ],
                                axis = 1)
#train_DF_weneed = train_DF.drop(['id'], axis = 1)
train_DF_weneed.info()

In [ ]:
#train_DF_weneed = train_DF.drop(['id'], axis = 1)
strCol2 = []
for col in train_DF_weneed.columns.values.tolist():
    if (not np.issubdtype(train_DF_weneed[col], np.int64)) and (not np.issubdtype(train_DF_weneed[col], np.float64)):
        strCol2.append(col)
        
strCol2

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
encoderLabel = LabelEncoder()

for sC in strCol2:
    train_DF_weneed[sC] = encoderLabel.fit_transform(train_DF_weneed[sC])

train_DF_weneed.head()

In [ ]:
#train_DF_weneed.reindex(columns = ['flag'] + xgb_feature_list)
df_x = train_DF_weneed.drop(['flag'], axis = 1)
df_y = train_DF_weneed['flag']
trainx, valx, trainy, valy = train_test_split(df_x, df_y, test_size=.2, random_state=2020)

# ??k?????? - ??????
cv = StratifiedKFold(n_splits=10, shuffle=True)

####### 
# new data
# tag: ?'edu_deg_cd', 'deg_cd', 'atdd_type',
# ~?\N???unknown
# job_year ?-1
# ???ind??????????
#      ?????????,
#      ??Car hou, ic fsind
#       ?ovd_30d_loan_tot_cnt, his_lng_ovd_day??
# ??????'dnl_mbl_bnk_ind', 'dnl_bind_cmb_lif_ind'
# ????????
# trx: ?????????????,???????????(????),?????,????
# ??beh: ???????????????,????
################# XGBoost
xgb_cfl = xgb.XGBClassifier(n_jobs = -1,
                            n_estimators = 50,
                            class_weight = 'balanced',
                            max_depth = 3)

xgb_cfl.fit(trainx, trainy)
y_scorexgb = xgb_cfl.predict_proba(valx)[:,1]

rocxgb=roc_auc_score(valy, y_scorexgb)
print('roc ',rocxgb)
perfomance_clf(xgb_cfl, valx, valy)

plot_roc(valy, y_scorexgb)

# ??XGB???????:0.7539461713230798
# 0.753749656824646
# 0.7539528921281328
# 0.754746591585147
# ? 'vld_rsk_ases_ind_1', 'vld_rsk_ases_ind_0', 'vld_rsk_ases_ind_-1', 'vld_rsk_ases_ind', 'total_page_no_tms', 'mrg_situ_cd_Z',
# 'mrg_situ_cd_Unknown', ##'mrg_situ_cd_B' 0.7539528921281328, 'mrg_situ_cd_A', 'loan_act_ind_1', 'loan_act_ind_0', 'loan_act_ind_-1',
# 'l6mon_agn_ind_1', 'l6mon_agn_ind_0', 'l6mon_agn_ind_-1', 'l6mon_agn_ind', 'l12_mon_insu_buy_whl_tms', 'l12_mon_gld_buy_whl_tms',
# 'ic_ind_1', 'ic_ind_0', 'ic_ind_-1', 'ic_ind', 'ic_fr_sh_ind', 'hav_hou_grp_ind_1', 'hav_hou_grp_ind_0', 'hav_hou_grp_ind_-1',
# 'hav_hou_grp_ind', 'hav_car_grp_ind_1', 'hav_car_grp_ind_-1', 'hav_car_grp_ind', 'gdr_cd_Unknown', 'gdr_cd_F', 'fr_or_sh_ind_1',
# 'fr_or_sh_ind_-1', 'fr_or_sh_ind', 'dnl_mbl_bnk_ind_1', 'dnl_mbl_bnk_ind_-1', 'dnl_bind_cmb_lif_ind_1', 'dnl_bind_cmb_lif_ind_-1',
# 'dnl_bind_cmb_lif_ind', 'cur_debit_crd_lvl', 'crd_card_act_ind_1', 'crd_card_act_ind_-1', 'crd_card_act_ind', 'confirm_rsk_ases_lvl_typ_cd',
# 'bk1_cur_year_mon_avg_agn_amt_cd', 'beh_page_no_ZY1', 'beh_page_no_SZD', 'beh_page_no_SYK', 'beh_page_no_LCT', 'beh_page_no_LC0',
# 'beh_page_no_JJD', 'beh_page_no_FLS', 'beh_page_no_FDA', 'beh_page_no_EGA', 'beh_page_no_CTR', 'beh_page_no_BWE', 'acdm_deg_cd_Z',
# 'acdm_deg_cd_Unknown', 'acdm_deg_cd_D', 'acdm_deg_cd_30',


##
# ??'Dat_Flg1_Cd_C', 'rsk_mean'  0.7561045164362351
# ?'Dat_Flg1_Cd_C' 0.754746591585147, 
# ?'Dat_Flg1_Cd_C', 'beh_page_no_MTA' 0.7550831381998223(??7531),
# STAR!! ?'Dat_Flg1_Cd_C', 'rsk_mean' 0.7561045164362351(??75430)
# ?'Dat_Flg1_Cd_C', 'beh_page_no_CQD' 0.7551685752832359 (??75221)
# ?'Dat_Flg1_Cd_C', 'mrg_situ_cd' 0.7558625214214152
# DatFlg1_Cd_C, 'frs_agn_dt_cnt': 0.7553716264550773, rsk_mean: 0.7561045164362351, ??725264
# DatFlg1_Cd_C, 'beh_page_no_CQE': 0.7547485249674225, rsk_mean: 0.7561045164362351, ??75395
# 0.7561285916488566


In [ ]:
train_x = train_DF_weneed.drop(['flag'], axis = 1)
train_y = train_DF_weneed.flag
colname = train_x.columns.values.tolist()

import xgboost as xgb
xgb_cfl = xgb.XGBClassifier(n_jobs = -1,
                            n_estimators = 50,
                            class_weight = 'balanced',
                            max_depth = 3)

xgb_cfl.fit(train_x, train_y)
xgb_feature = sorted(zip(map(lambda x: round(x, 4), xgb_cfl.feature_importances_), colname),  reverse=True)
xgb_feature

In [ ]:
print(len(xgb_feature))
xgb_feature_list = [x[1] for x in xgb_feature]
xgb_feature_score = [x[0] for x in xgb_feature]
#xgb_feature_list[xgb_feature_score.index(0.0)-1:]
xgb_feature_list

In [ ]:
import xgboost as xgb
from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression
from sklearn.ensemble import RandomForestClassifier ,RandomForestRegressor
from sklearn.model_selection import KFold,train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score,accuracy_score
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve, precision_score, recall_score
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn import model_selection
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

def perfomance_clf(model,X,y,name=None):
    y_predict = model.predict(X)
    y_predict_prob = model.predict_proba(X)[:, 1]
    if name:
        print(name,':')
    print(f'accuracy score is: {accuracy_score(y,y_predict)}')
    print(f'precision score is: {precision_score(y,y_predict)}')
    print(f'recall score is: {recall_score(y,y_predict)}')
    print(f'auc: {roc_auc_score(y,y_predict_prob)}')
    print('- - - - - - ')
    
def plot_roc(y_test,prob):
    fpr, tpr, thresholds = roc_curve(y_test, prob)
    # plot no skill
    plt.plot([0, 1], [0, 1], linestyle='--')
    # plot the roc curve for the model
    plt.plot(fpr, tpr)#, marker='.')
    plt.title("ROC curve")
    plt.xlabel('false positive rate')
    plt.ylabel('true positive rate')
    # show the plot
    plt.show()    

In [ ]:
#train_DF_weneed.reindex(columns = ['flag'] + xgb_feature_list)
df_x = train_DF_weneed.drop(['flag'], axis = 1)
df_y = train_DF_weneed['flag']
trainx, valx, trainy, valy = train_test_split(df_x, df_y, test_size=.2, random_state=2020)

# ??k?????? - ??????
cv = StratifiedKFold(n_splits=10, shuffle=True)

In [ ]:
####### 
# ?XGB????,??'mrg_situ_cd', 'ovd_30d_loan_tot_cnt'
# new data
# tag: ?'edu_deg_cd', 'deg_cd', 'atdd_type',
# ~?\N???unknown
# job_year ?-1
#      ?????????,
#      ??Car hou, ic fsind
#       ?ovd_30d_loan_tot_cnt, his_lng_ovd_day??
# ??????'dnl_mbl_bnk_ind', 'dnl_bind_cmb_lif_ind'
# ????????
# trx: ?????????????,???????????(????),?????,????
# ??beh: ???????????????,????
################# XGBoost
xgb_cfl = xgb.XGBClassifier(n_jobs = -1,
                            n_estimators = 50,
                            class_weight = 'balanced',
                            max_depth = 3)

xgb_cfl.fit(trainx, trainy)
y_scorexgb = xgb_cfl.predict_proba(valx)[:,1]

rocxgb=roc_auc_score(valy, y_scorexgb)
print('roc ',rocxgb)
perfomance_clf(xgb_cfl, valx, valy)

plot_roc(valy, y_scorexgb)

In [ ]:
####### 
# ?XGB????,??'mrg_situ_cd', 'ovd_30d_loan_tot_cnt'
# new data
# tag: ?'edu_deg_cd', 'deg_cd', 'atdd_type',
# ~?\N???unknown
# job_year ?-1
#      ?????????,
#      ??Car hou, ic fsind
#       ?ovd_30d_loan_tot_cnt, his_lng_ovd_day??
# ??????'dnl_mbl_bnk_ind', 'dnl_bind_cmb_lif_ind'
# ????????
# trx: ?????????????,???????????(????),?????,????
# ??beh: ???????????????,????
################# XGBoost
xgb_cfl = xgb.XGBClassifier(n_jobs = -1,
                            n_estimators = 50,
                            class_weight = 'balanced',
                            max_depth = 3)

xgb_cfl.fit(trainx, trainy)
y_scorexgb = xgb_cfl.predict_proba(valx)[:,1]

rocxgb=roc_auc_score(valy, y_scorexgb)
print('roc ',rocxgb)
perfomance_clf(xgb_cfl, valx, valy)

plot_roc(valy, y_scorexgb)

In [ ]:
# ?????????,'????????'????????,?????????
def resample(df):
    '''
    ???'????????'???0,1???????,????????sample()??????random_state?????????
    df:???
    '''
    num = df['flag'].value_counts()[1]
    df_t = df[df.flag == 1]
    df_f = df[df.flag == 0].sample(frac=1)[0:num]
    df_balanced = pd.concat([df_t,df_f]).sample(frac=1).reset_index(drop=True)
#     print(df_balanced.????????.value_counts())
    return df_balanced

df_balanced = resample(train_DF_weneed)
df_balanced['flag'].value_counts()